# Notebook 1 — Selective Frequency Reconstruction

---

## What is K-Space?

When an MRI scanner takes a scan, it doesn't capture pixels directly. Instead, it records **frequency waves** — mathematical signals that encode where tissue is and how dense it is. This raw frequency data is called **k-space**.

Think of it like music:
- A song is made of many frequencies (bass, mid, treble) layered together
- K-space is the raw frequency data of a body part
- The image you see on a screen is what you get *after* the scanner decodes those frequencies

The decoding step is called the **Inverse Fast Fourier Transform (IFFT)**. It converts frequency data → pixel image.

---

## What Does K-Space Look Like?

K-space is a 2D grid. The **center** of that grid holds low-frequency information. The **edges** hold high-frequency information.

```
K-Space Grid:

  edges  →  high frequency  →  fine detail, sharp edges
  center →  low frequency   →  coarse shape, bulk contrast
```

Most MRI software uses ALL of k-space to build the image. **This notebook uses only part of it** — and shows you what each part contains.

---

## The Core Idea: Selective Reconstruction

Instead of applying IFFT to the full k-space, we apply a **mask** first — zeroing out regions we don't want — then run IFFT on what remains.

```
Full k-space  →  apply mask  →  masked k-space  →  IFFT  →  partial image
```

This lets us separate tissue types by their frequency content — **without any segmentation algorithm**. No AI, no manual tracing. Just physics.

---

## Three Masks We'll Use

| Mask | What it keeps | What you see in the image |
|---|---|---|
| **Low frequency** | Center of k-space only | Blurry but correct overall shape — bones, large tissue volumes |
| **High frequency** | Edges of k-space only | Sharp edges and fine detail — cartilage boundaries, surface texture |
| **Band mask** | A ring between inner and outer radius | Intermediate structures — specific tissue layers |

---

## What is RSS (Root Sum of Squares)?

MRI scanners often use multiple receiver coils — like having multiple microphones recording at once. Each coil records its own version of k-space. To combine them into one image, we:

1. Apply IFFT to each coil separately → get one image per coil
2. Square each pixel value per coil
3. Sum across all coils
4. Take the square root

This is called **Root Sum of Squares (RSS)**. It's the standard way to merge multi-coil data without needing to know coil geometry.

> The knee single-coil dataset has only 1 coil, so RSS just applies IFFT and takes the absolute value. The code handles both cases automatically.

---

## Step 1 — Import Libraries

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from kode.io import load_kspace, root_sum_of_squares
from kode.selective_reconstruct import (
    low_frequency_mask,
    high_frequency_mask,
    band_mask,
    selective_reconstruct,
)

---

## Step 2 — Load a K-Space Slice

The fastMRI `.h5` file contains a full 3D knee scan stored as a stack of 2D slices. Each slice is one cross-section of the knee.

- `H5_FILE` — path to the data file
- `SLICE_IDX` — which cross-section to load (0 = first slice, 15 = middle of the scan)

The k-space array has shape `(coils, height, width)`. For single-coil data, coils = 1.

In [ ]:
H5_FILE = '../data/singlecoil_test/file1000022.h5'
SLICE_IDX = 15

kspace = load_kspace(H5_FILE, slice_idx=SLICE_IDX)
print(f'K-space shape: {kspace.shape}  →  (coils, height, width)')
print(f'Data type: {kspace.dtype}  →  complex numbers (real + imaginary parts)')

---

## Step 3 — Reconstruct with Different Masks

Here we build four images from the same k-space slice:

1. **Full** — no mask, standard reconstruction (the ground truth image)
2. **Low frequency only** — keep the central 8% of k-space
3. **High frequency only** — remove the central 8%, keep the edges
4. **Band mask** — keep frequencies between 5% and 25% from center

The `fraction` parameter controls what percentage of k-space is kept. Smaller = more selective.

In [ ]:
_, H, W = kspace.shape

full_image  = root_sum_of_squares(kspace)
low_image   = selective_reconstruct(kspace, low_frequency_mask(H, W, fraction=0.08))
high_image  = selective_reconstruct(kspace, high_frequency_mask(H, W, fraction=0.08))
band_image  = selective_reconstruct(kspace, band_mask(H, W, inner=0.05, outer=0.25))

fig, axes = plt.subplots(1, 4, figsize=(18, 5))
titles = [
    'Full Reconstruction\n(all frequencies)',
    'Low Freq Only\n(coarse structure)',
    'High Freq Only\n(edges + detail)',
    'Band 5–25%\n(intermediate tissue)',
]
images = [full_image, low_image, high_image, band_image]

for ax, img, title in zip(axes, images, titles):
    ax.imshow(img, cmap='gray')
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle('Selective Frequency Reconstruction — Kode', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('../results/selective_reconstruct.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved to results/selective_reconstruct.png')

---

## Step 4 — Visualize the Masks Themselves

These are the actual binary grids being applied to k-space before reconstruction. White = kept, black = zeroed out.

Notice:
- The low-frequency mask is a small square in the center
- The high-frequency mask is the inverse — everything except the center
- The band mask is a ring (annulus) at a specific distance from center

The shape of these masks is what determines which tissue structures appear in the reconstructed image.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
masks = [
    low_frequency_mask(H, W, fraction=0.08),
    high_frequency_mask(H, W, fraction=0.08),
    band_mask(H, W, inner=0.05, outer=0.25),
]
mask_titles = [
    'Low Freq Mask\n(center square)',
    'High Freq Mask\n(edges only)',
    'Band Mask\n(annular ring)',
]

for ax, mask, title in zip(axes, masks, mask_titles):
    ax.imshow(mask, cmap='viridis')
    ax.set_title(title, fontsize=11)
    ax.axis('off')

plt.suptitle('K-Space Masks — What Gets Kept Before IFFT', fontsize=13)
plt.tight_layout()
plt.show()

---

## What This Means

Traditional MRI pipelines treat reconstruction as a single step — full k-space in, full image out. Kode shows that **reconstruction is a choice**, not a fixed operation.

By selecting which frequencies to include, you can:
- Isolate cartilage boundaries without segmentation
- Separate soft tissue from bone by their frequency content
- Reduce noise by filtering out high-frequency artifacts
- Ask: *how little k-space do I actually need to make a useful image?*

This is the foundation of **compressed sensing** — a technique used in modern fast MRI scanners that reconstructs full images from only 25–30% of k-space data.